# 16 - Gold Revenue Daily

Computes revenue by status-change date and shipping country. `paid` adds revenue; `cancelled` and `returned` subtract it.

This notebook is deterministic and safe to rerun. It synchronizes
the complete business result with Delta `MERGE`, records an audit
run, and refreshes aggregate-to-source lineage.


In [ ]:
%run ./05_bronze_common


In [ ]:
import uuid

from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import functions as F


dbutils.widgets.text("catalog", "abc_retail", "Target catalog")
dbutils.widgets.text("job_run_id", "manual", "Notebook or job run ID")

catalog = validate_identifier(dbutils.widgets.get("catalog").strip(), "catalog")
job_run_id = dbutils.widgets.get("job_run_id").strip()
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`gold`")


def sync_gold_table(target_table, source_df, keys):
    """Synchronize a deterministic Gold snapshot with an idempotent Delta MERGE."""
    source_df = source_df.dropDuplicates(keys)
    target_df = spark.table(target_table)
    stale_keys = (
        target_df.select(*keys)
        .join(source_df.select(*keys), keys, "left_anti")
        .withColumn("_delete", F.lit(True))
    )
    merge_source = (
        source_df.withColumn("_delete", F.lit(False))
        .unionByName(stale_keys, allowMissingColumns=True)
    )
    condition = " AND ".join(
        f"target.`{column}` = source.`{column}`" for column in keys
    )
    assignments = {
        column: f"source.`{column}`" for column in source_df.columns
    }
    (
        DeltaTable.forName(spark, target_table)
        .alias("target")
        .merge(merge_source.alias("source"), condition)
        .whenMatchedDelete(condition="source._delete")
        .whenMatchedUpdate(
            condition=(
                "NOT source._delete "
                "AND NOT (target.record_hash <=> source.record_hash)"
            ),
            set=assignments,
        )
        .whenNotMatchedInsert(
            condition="NOT source._delete",
            values=assignments,
        )
        .execute()
    )


def replace_gold_lineage(target_table, lineage_df):
    """Keep the current direct source mappings for one aggregate Gold table."""
    lineage_target = DeltaTable.forName(
        spark, f"{catalog}.control.record_lineage"
    )
    lineage_target.delete(F.col("target_table") == F.lit(target_table))
    prepared = (
        lineage_df.select(
            F.lit(target_table).alias("target_table"),
            F.col("target_record_key").cast("string"),
            F.lit(pipeline_run_id).alias("pipeline_run_id"),
            "source_table",
            F.col("source_record_key").cast("string"),
            "source_file",
            "batch_id",
            F.col("source_ingest_ts").cast("timestamp"),
            F.current_timestamp().alias("recorded_ts"),
        )
        .dropDuplicates(
            [
                "target_table",
                "target_record_key",
                "source_table",
                "source_record_key",
                "source_file",
                "batch_id",
            ]
        )
    )
    if prepared.limit(1).count():
        prepared.write.format("delta").mode("append").saveAsTable(
            f"{catalog}.control.record_lineage"
        )


In [ ]:
pipeline_name = "gold_revenue_daily"
pipeline_run_id = str(uuid.uuid4())
start_ts = datetime.now(timezone.utc)
source_table = f"{catalog}.silver.silver_order_status_history"
target_table = f"{catalog}.gold.gold_revenue_daily"

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS `{catalog}`.`gold`.`gold_revenue_daily` (
        revenue_date DATE NOT NULL,
        country STRING NOT NULL,
        gross_paid_usd DECIMAL(28,2) NOT NULL,
        reversed_usd DECIMAL(28,2) NOT NULL,
        net_revenue_usd DECIMAL(28,2) NOT NULL,
        paid_events BIGINT NOT NULL,
        reversal_events BIGINT NOT NULL,
        affected_orders BIGINT NOT NULL,
        record_hash STRING NOT NULL,
        updated_ts TIMESTAMP NOT NULL,
        pipeline_run_id STRING NOT NULL
    )
    USING DELTA
    PARTITIONED BY (revenue_date)
    TBLPROPERTIES (
        'comment' = 'Daily country revenue using paid and reversal status timeline'
    )
    """
)

upsert_audit(
    catalog, pipeline_run_id, pipeline_name, "full_snapshot", start_ts,
    "RUNNING", [source_table], job_run_id,
)

try:
    history = (
        spark.table(source_table)
        .filter(F.col("status").isin("paid", "cancelled", "returned"))
        .withColumn("revenue_date", F.to_date("status_ts"))
        .withColumn("country", F.coalesce("country", F.lit("UNKNOWN")))
    )
    business = (
        history.groupBy("revenue_date", "country")
        .agg(
            F.sum(
                F.when(F.col("status") == "paid", F.col("subtotal_usd"))
                .otherwise(F.lit(0))
            ).cast("decimal(28,2)").alias("gross_paid_usd"),
            F.sum(
                F.when(
                    F.col("status").isin("cancelled", "returned"),
                    F.col("subtotal_usd"),
                ).otherwise(F.lit(0))
            ).cast("decimal(28,2)").alias("reversed_usd"),
            F.sum(F.when(F.col("status") == "paid", 1).otherwise(0))
            .cast("long").alias("paid_events"),
            F.sum(
                F.when(
                    F.col("status").isin("cancelled", "returned"), 1
                ).otherwise(0)
            ).cast("long").alias("reversal_events"),
            F.countDistinct("order_id").cast("long").alias("affected_orders"),
        )
        .withColumn(
            "net_revenue_usd",
            (F.col("gross_paid_usd") - F.col("reversed_usd"))
            .cast("decimal(28,2)"),
        )
    )
    source = (
        business.withColumn(
            "record_hash",
            F.sha2(
                F.concat_ws(
                    "||",
                    F.col("gross_paid_usd"),
                    F.col("reversed_usd"),
                    F.col("net_revenue_usd"),
                    F.col("paid_events"),
                    F.col("reversal_events"),
                    F.col("affected_orders"),
                ),
                256,
            ),
        )
        .withColumn("updated_ts", F.current_timestamp())
        .withColumn("pipeline_run_id", F.lit(pipeline_run_id))
    )
    sync_gold_table(target_table, source, ["revenue_date", "country"])

    lineage = history.select(
        F.concat_ws("|", F.col("revenue_date"), F.col("country"))
        .alias("target_record_key"),
        F.lit(source_table).alias("source_table"),
        F.concat_ws("|", "order_id", "status", "status_ts")
        .alias("source_record_key"),
        "source_file",
        "batch_id",
        F.col("ingest_ts").alias("source_ingest_ts"),
    )
    replace_gold_lineage(target_table, lineage)

    upsert_audit(
        catalog, pipeline_run_id, pipeline_name, "full_snapshot", start_ts,
        "SUCCESS", [source_table], job_run_id,
        end_ts=datetime.now(timezone.utc), rows_read=history.count(),
        rows_written=source.count(), rows_quarantined=0,
    )
except Exception as exc:
    upsert_audit(
        catalog, pipeline_run_id, pipeline_name, "full_snapshot", start_ts,
        "FAILED", [source_table], job_run_id,
        end_ts=datetime.now(timezone.utc), error_message=str(exc)[:4000],
    )
    raise

spark.table(target_table).orderBy("revenue_date", "country").display()
